In [6]:
# Create a database
from pymongo import MongoClient
from dotenv import dotenv_values

config = dotenv_values(".env")

def get_database():
    client = MongoClient(config["MONGO_DB_URL"])
    return client['nikodem']

database = get_database()

In [7]:
# Remove old data
database["salaries"].delete_many({})

DeleteResult({'n': 46, 'ok': 1.0}, acknowledged=True)

In [8]:
import csv

salaries = []

with open('salaries.csv', 'r') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:        
        transferred_at = row[2].split('.')
                    
        salaries.append({
            "year": row[0],
            "month": row[1],
            "transferred_at": f'{transferred_at[2]}-{transferred_at[1]}-{transferred_at[0]}',
            "amount": float(row[3][1:].replace(',', '')),
            "currency": config["CURRENCY"],
            "employer": config["EMPLOYER"],
        })


In [9]:
from dateutil import parser
from datetime import datetime
import requests

def fetch_rate_from_nbp(date, currency) -> requests.Response:
    return requests.get(f'http://api.nbp.pl/api/exchangerates/rates/a/{currency}/{date}')

def insert_salary(salary):
    date_of_transfer = salary['transferred_at']
    
    rate = fetch_rate_from_nbp(date_of_transfer, salary["currency"]).json()['rates'][0]['mid']
    
    collection = database['salaries']

    salary = {
        "month": int(salary['month']),
        "year": int(salary['year']),
        "salary": salary['amount'],
        "currency": salary['currency'],
        "rate": rate,
        "employer": salary['employer'],
        "transferred_at": parser.parse(date_of_transfer),
        "created_at":  parser.parse(datetime.now().isoformat())
    }

    collection.insert_one(salary)

for salary in salaries:
    insert_salary(salary)